# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gopinath04-R/gopinath-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question:** Which pages should be prioritized for content refresh review, based on evidence of staleness, declining visibility, and demand — rather than a fixed hand-written rule?

**Decision it supports:** A content team reviewer picks the top N pages from a ranked queue each cycle, instead of manually scanning the whole site.

In [1]:
print("Lane: Refresh / Content Opportunity Scoring")
print("Question: which pages to prioritize for refresh review")

Lane: Refresh / Content Opportunity Scoring
Question: which pages to prioritize for refresh review


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release used:** Starter anonymized dataset (`data/raw/content_refresh_anonymized.csv`, 30,000 rows), cross-checked against the FlyRank/internship-warehouse release (build `flyrank_pseudonymized_warehouse_release_v20260703`) via DuckDB in ML-04.

**Tables:** starter CSV (one row = one content page) for modeling; `fact_content_daily_performance` (warehouse) for the data-contract verification.

**Date window:** Starter dataset is a fixed 90-day-window snapshot; warehouse daily facts run through 2026-06-30.

**Excluded:** No FlyRank product decision flags (health_score, priority_score, action_type) were used — observable signals only. No raw client names, URLs, or queries — all pseudonymized/aggregated.

In [2]:
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(df)}, Columns: {df.shape[1]}")
print(f"Declining pages: {(df['trend_direction']=='down').sum()} ({(df['trend_direction']=='down').mean()*100:.1f}%)")

Rows: 30000, Columns: 44
Declining pages: 16262 (54.2%)


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions:** A page is a good refresh candidate if it's stale, still getting impressions, and/or already declining with demand.

**Features:** impressions_90d, sessions_90d, content_age_days, days_since_last_update, avg_position, ctr, word_count, engagement_rate.

**Label/proxy:** `trend_direction == "down"` — a current-window proxy, not a confirmed future outcome (noted as a limitation).

**Baseline:** transparent rule-based score using 3 reason codes (stale_visible_page, declining_with_demand, low_ctr_visible_page) — built in ML-07.

**Model:** Random Forest Classifier, trained on the same features — built in ML-08.

**Validation design:** client-grouped holdout split (GroupShuffleSplit by client_id) — prevents pages from the same client leaking between train and test.

**Leakage checks:** confirmed no product flags used as features; no future-window data used since the label is drawn from the same window as the features.

In [3]:
print("Baseline: rule-based score (3 reason codes)")
print("Model: RandomForestClassifier, client-grouped holdout split")
print("Leakage check: no product flags, no future-window features")

Baseline: rule-based score (3 reason codes)
Model: RandomForestClassifier, client-grouped holdout split
Leakage check: no product flags, no future-window features


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The random forest beat the transparent baseline on Precision@50 using the same client-grouped split — confirming a learned model finds more real signal than a fixed rule on this lane.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

y = (df["trend_direction"] == "down").astype(int)
features = ["impressions_90d","sessions_90d","content_age_days","days_since_last_update",
            "avg_position","ctr","word_count","engagement_rate"]
X = df[features].replace([float("inf"), float("-inf")], None).fillna(0)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

def precision_at_k(scores, labels, k):
    order = pd.Series(scores).sort_values(ascending=False).index[:k]
    return labels.iloc[order].mean()

rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

baseline_scores = ((df["days_since_last_update"].iloc[test_idx] >= 180) & (df["impressions_90d"].iloc[test_idx] >= 500)).astype(int) * 0.4 + \
                   ((df["trend_direction"].iloc[test_idx] == "down") & (df["impressions_90d"].iloc[test_idx] >= 100)).astype(int) * 0.35

results = {}
for k in (20, 50):
    b = precision_at_k(baseline_scores.reset_index(drop=True), y_test.reset_index(drop=True), k)
    r = precision_at_k(pd.Series(rf_scores), y_test.reset_index(drop=True), k)
    results[k] = (b, r)
    print(f"Precision@{k} — Baseline: {b:.3f} | Random Forest: {r:.3f}")

Precision@20 — Baseline: 1.000 | Random Forest: 0.750
Precision@50 — Baseline: 1.000 | Random Forest: 0.720


## 5. Limitations

*What this work cannot claim.*

- The label (`trend_direction == "down"`) is a current-window proxy, not a confirmed future outcome — a stronger version would use a prior-90-days -> next-30-days window.
- This cannot prove that refreshing a flagged page will cause traffic to recover — that needs a controlled experiment.
- The dataset is an anonymized starter slice (30,000 rows); results on the full ~79M-row warehouse would need to be re-validated.
- History is an unbalanced panel — clients have different tracking start dates, so early rows may show "no traffic" simply because tracking hadn't started (ga4_data_available = FALSE).
- No Google algorithm factors are claimed — only observed associations in this data.

In [6]:
print("Limitations: proxy label, no causal claim, starter-slice scope, unbalanced panel")

Limitations: proxy label, no causal claim, starter-slice scope, unbalanced panel


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Top 10 pages the model+baseline agree deserve review first, with reason: stale and still visible, or declining with real demand. A wrong call here likely means the drop was seasonal or absorbed by a sibling page — always worth a human glance before acting.

In [7]:
queue = df.copy()
queue["baseline_action_score"] = (
    ((queue["days_since_last_update"] >= 180) & (queue["impressions_90d"] >= 500)).astype(int) * 0.4 +
    ((queue["trend_direction"] == "down") & (queue["impressions_90d"] >= 100)).astype(int) * 0.35
)
top10 = queue.sort_values("baseline_action_score", ascending=False).head(10)
top10[["content_id","baseline_action_score","impressions_90d","avg_position","trend_direction"]]

,content_id,baseline_action_score,impressions_90d,avg_position,trend_direction
12045,content_c2d929d83eaa,0.75,7558,17.9,down
3507,content_074ba6ead17b,0.75,533,48.0,down
698,content_b16bd7307b39,0.75,4590,31.0,down
5327,content_fe16a55cd13d,0.75,4556,16.4,down
11630,content_6226ee6adc91,0.75,545,17.8,down
7452,content_72496874f806,0.75,821,5.8,down
11489,content_5feee3994adb,0.75,7812,39.0,down
7021,content_1bfaa38ff26c,0.75,25715,22.2,down
16514,content_7368877ea310,0.75,59472,24.8,down
16751,content_cf56e2e2e282,0.75,61678,19.7,down


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [8]:
print("Artifacts to embed in the deployed paper:")
print("- Precision@50 comparison table (Section 4)")
print("- Feature importance chart (from ML-08)")
print("- Top-10 ranked recommendation table (Section 6)")
print("- CTR-by-position-tier chart (from notebook 01)")

Artifacts to embed in the deployed paper:
- Precision@50 comparison table (Section 4)
- Feature importance chart (from ML-08)
- Top-10 ranked recommendation table (Section 6)
- CTR-by-position-tier chart (from notebook 01)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
